# 3. Применение requests to add

По умолчанию используется безопасный `review`-режим: решения записываются в лог, но каталог не меняется. После проверки результатов установите `APPLY_MODE = "auto"`; будут применены только решения с confidence не ниже `AUTO_APPLY_MIN_CONFIDENCE`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
%cd {ROOT}


In [ ]:
from src.config import Settings
from src.file_io import load_catalog, load_jsonl
from src.request_merger import run_request_merge

settings = Settings.from_env()
requests_path = settings.artifacts_dir / "inference" / "add_requests.jsonl"
requests = [item for item in load_jsonl(requests_path) if item.get("status") == "pending"]
print(f"Pending requests: {len(requests)}")
print(f"Auto threshold: {settings.auto_apply_min_confidence}")


In [ ]:
# Сначала запустите review. Поменяйте значение на "auto" только после просмотра merge_decisions.jsonl.
APPLY_MODE = "review"
assert APPLY_MODE in {"review", "auto"}

catalog = run_request_merge(settings, mode=APPLY_MODE)
print(f"Catalog v{catalog.catalog_version}: {len(catalog.drivers)} drivers")


In [ ]:
import pandas as pd
from pathlib import Path

log_path = settings.artifacts_dir / "merge" / "merge_decisions.jsonl"
if log_path.exists():
    pd.read_json(log_path, lines=True)[["application_status", "catalog_version_before", "catalog_version_after"]]
else:
    print("No merge decisions were generated.")
